# Lưu ý khi chạy code: 

Ai dùng thì ctrl F khúc này: 

    urls = urls[:100] 

    # urls = urls[0:100]   # Hiếu

    # urls = urls[100:200] # Lợi

    # urls = urls[200:300] # Thiện

    # urls = urls[300:400] # Thịnh

comment dòng đầu, mở comment dòng của mình r chạy 

* Lúc cào mở màn hình chrome cào lên, để nửa màn hình cũng được, ko mở thì nó ít hiện thông tin

* Nếu lỗi gì ko cào được đủ hết hoặc cào bị thiếu thì chờ 10-15p sau hẵng chạy code lại,  do ggmap phát hiện bot nên block, cào lại liền nó ko hiện gì cho cào.



In [5]:
!pip install undetected-chromedriver


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import re
import time
import random
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options


import undetected_chromedriver as uc

def init_driver():
    global driver  # QUAN TRỌNG: Ép các hàm khác dùng chung biến này
    print("   🌐 Đang khởi tạo cửa sổ Chrome mới...")
    options = uc.ChromeOptions()
    options.add_argument("--lang=vi-VN")
    #options.add_argument("--start-maximized")
    
    # Khởi tạo driver
    driver = uc.Chrome(options=options, version_main=146)
    return driver


def safe_get_text(xpath):
    try:
        return driver.find_element(By.XPATH, xpath).text.strip()
    except:
        return None


def get_basic_info():
    data = {"name": None, "avg_rating": None, "review_count": None}
    try:
        # 1. Chờ H1 xuất hiện (Tên địa điểm)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.TAG_NAME, "h1"))
        )
        
        # 2. CHỜ ĐỢI CHIẾN THUẬT: Chờ 2 giây để Google Maps load nốt Rating và các Tab
        print("   ⏳ Đang đợi Google Maps render đủ thông tin (2s)...")
        time.sleep(2) 
        
        # Lấy tên sau khi đã đợi
        data["name"] = driver.find_element(By.TAG_NAME, "h1").text.strip()

        # 3. Lấy Rating và Review Count (Dùng Selector linh hoạt hơn)
        # Thử tìm thẻ div chứa class F7nice (nơi chứa rating)
        try:
            stats_el = driver.find_element(By.CLASS_NAME, "F7nice")
            full_text = stats_el.get_attribute("innerText").replace(".", "").replace(",", "")
            nums = re.findall(r'\d+', full_text)
            
            if len(nums) >= 1: data["avg_rating"] = nums[0]
            if len(nums) >= 2: data["review_count"] = nums[1]
        except:
            print("   ⚠️ Không tìm thấy cụm Rating (có thể quán chưa có đánh giá).")

    except Exception as e:
        print(f"   ⚠️ Lỗi lấy Basic Info: {e}")
        
    return data

# ===== CLICK REVIEW TAB (có retry) =====
def click_reviews_tab():
    for _ in range(2):
        try:
            btn = WebDriverWait(driver, 5).until(
                EC.presence_of_element_located(
                    (By.XPATH,
                     "//button[contains(@aria-label, 'Review') or contains(@aria-label, 'Bài đánh giá')]")
                )
            )
            driver.execute_script("arguments[0].click();", btn)
            time.sleep(2)
            return True
        except:
            driver.refresh()
            time.sleep(3)
    return False



def scroll_reviews(scroll_box, max_scroll=50):
    last_height = 0

    for _ in range(max_scroll):
        driver.execute_script(
            "arguments[0].scrollTop = arguments[0].scrollHeight", scroll_box
        )
        time.sleep(random.uniform(1.5, 2.5))

        new_height = driver.execute_script(
            "return arguments[0].scrollHeight", scroll_box
        )

        if new_height == last_height:
            break

        last_height = new_height

def get_reviews(max_reviews=50):
    results = []
    seen_reviewers = set()
    
    # 1. Tìm khung cuộn review (Scroll Box)
    try:
        scroll_box = WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.XPATH, "//div[@role='main' and contains(@aria-label, 'đánh giá')] | //div[contains(@class, 'm6QErb') and contains(@class, 'DxyBCb')]"))
        )
    except:
        try:
            # Dự phòng nếu Xpath trên tạch: Tìm từ 1 review block ngược lên cha nó
            sample = driver.find_element(By.CLASS_NAME, "jftiEf")
            scroll_box = sample.find_element(By.XPATH, "./..")
        except:
            print("   ⚠️ Không tìm thấy khung cuộn review.")
            return []

    no_new_count = 0
    while len(results) < max_reviews:
        # Lấy danh sách block đang hiển thị
        review_blocks = driver.find_elements(By.CLASS_NAME, "jftiEf")
        current_len = len(results)
        
        for r in review_blocks:
            if len(results) >= max_reviews: break
            try:
                # Lấy tên (dùng textContent để tránh rỗng khi chưa render hết)
                name = r.find_element(By.CLASS_NAME, "d4r55").get_attribute("textContent").strip()
                
                if name and name not in seen_reviewers:
                    # Cuộn tới review này để ép Google nạp data
                    driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", r)
                    
                    # Bấm "Xem thêm"
                    try:
                        more_btn = r.find_element(By.XPATH, ".//button[contains(., 'Xem thêm')]")
                        driver.execute_script("arguments[0].click();", more_btn)
                    except: pass

                    # Cào các thông tin chính
                    rating_label = r.find_element(By.CLASS_NAME, "kvMYJc").get_attribute("aria-label")
                    rating = re.search(r'\d+', rating_label).group() if rating_label else "0"
                    time_text = r.find_element(By.CLASS_NAME, "rsqaWe").get_attribute("textContent").strip()
                    
                    comment = ""
                    try:
                        comment = r.find_element(By.CLASS_NAME, "wiI7pd").get_attribute("textContent").strip()
                    except: pass

                    # --- PHẦN CÀO FEATURE (Đồ ăn, Dịch vụ...) ---
                    features = {}
                    try:
                        # Tìm các thẻ chứa feature (class RfDO5c)
                        feature_els = r.find_elements(By.CLASS_NAME, "RfDO5c")
                        for fe in feature_els:
                            txt = fe.get_attribute("textContent").strip()
                            if ":" in txt:
                                k, v = txt.split(":", 1)
                                features[k.strip()] = v.strip()
                    except: pass

                    # Lưu kết quả
                    results.append({
                        "reviewer": name,
                        "rating": rating,
                        "time": time_text,
                        "comment": comment,
                        "features": features
                    })
                    seen_reviewers.add(name)
            except: continue

        # 2. Logic cuộn xuống để kích hoạt load thêm review mới
        driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_box)
        time.sleep(random.uniform(2.5, 4.0)) # Chờ Google Maps gọi API lấy thêm data

        # 3. Kiểm tra xem có lấy thêm được cái nào không
        if len(results) == current_len:
            no_new_count += 1
            # Thử "nhích" lên rồi xuống lại để đánh thức sự kiện scroll
            driver.execute_script("arguments[0].scrollTop -= 500", scroll_box)
            time.sleep(1)
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_box)
            
            if no_new_count >= 3: # Sau 3 lần thử mà vẫn ko có thêm thì thôi
                break
        else:
            no_new_count = 0 # Reset nếu có data mới

    return results






def normalize_url(url):
    # nếu có hl=en → đổi thành hl=vi
    if "hl=en" in url:
        url = url.replace("hl=en", "hl=vi")
    elif "hl=" not in url:
        # nếu chưa có hl → thêm vào
        if "?" in url:
            url += "&hl=vi"
        else:
            url += "?hl=vi"

    return url

# ===== MAIN (có retry + ép hl=vi) =====
def scrape_place(url):
    url = normalize_url(url)
    for _ in range(2):  # retry toàn bộ nếu bị block

        driver.get(url)
        time.sleep(random.uniform(3, 5))

        data = get_basic_info()

        if click_reviews_tab():
            data["reviews"] = get_reviews(50)
            return data
        else:
            print("⚠️ Không thấy Reviews → retry lại...")
            time.sleep(3)

    data["reviews"] = []
    return data



In [ ]:
import json
import os


# --- CẤU HÌNH ---
FILE_PATH = "../url_ngoai_quan_5.txt"
OUTPUT_FILE = "google_maps_results.json" 
BATCH_SIZE = 100
BREAK_TIME = 10
WARMUP_URL = "https://www.google.com/maps/place/PH%E1%BB%9E+HI%E1%BB%80N+%7C+PH%E1%BB%9E+NGON+QU%E1%BA%ACN+5/data=!4m7!3m6!1s0x31752fb69dc45ddf:0xc9dea8c16b465c48!8m2!3d10.7579661!4d106.6714435!16s%2Fg%2F11t7ysr25z!19sChIJ313EnbYvdTERSFxGa8Go3sk?authuser=0&hl=en&rclk=1"
# --- ĐỌC DANH SÁCH URL ---
try:
    with open(FILE_PATH, "r", encoding="utf-8") as f:
        urls = [line.strip() for line in f if line.strip()]


        urls = urls[:100] 

        # urls = urls[0:100]   # Hiếu
        # urls = urls[100:200] # Lợi
        # urls = urls[200:300] # Thiện
        # urls = urls[300:400] # Thịnh
                           
    print(f"✅ Đã tải {len(urls)} URLs.")
except FileNotFoundError:
    print(f"❌ Không tìm thấy file: {FILE_PATH}")
    urls = []

# Xóa file cũ để ghi mới
if os.path.exists(OUTPUT_FILE):
    os.remove(OUTPUT_FILE)

all_results = []
current_batch_count = 0
driver = None # Khởi tạo biến rỗng

for index, url in enumerate(urls):
    
    # 1. LOGIC KHỞI TẠO / RESET & WARM-UP
    # Chạy khi: Bắt đầu (index=0) HOẶC Khi đủ Batch
    if index == 0 or current_batch_count >= BATCH_SIZE:
        if index != 0:
            print(f"\n☕ Đã cào {BATCH_SIZE} link. Tiến hành reset trình duyệt...")
            try: driver.quit()
            except: pass
            time.sleep(BREAK_TIME)
        
        # Khởi tạo driver mới
        print("🚀 Đang khởi tạo trình duyệt...")
        init_driver() 
        driver.maximize_window()
        
        # --- BƯỚC WARM-UP (QUAN TRỌNG) ---
        print(f"🔥 Đang chạy Warm-up cho trình duyệt: {WARMUP_URL}")
        try:
            _ = scrape_place(WARMUP_URL) # Cào mồi, không lưu
            print("   ✅ Trình duyệt đã 'nóng', bắt đầu cào thật...")
        except:
            print("   ⚠️ Warm-up lỗi nhưng vẫn tiếp tục...")
            
        current_batch_count = 0

    # 2. TIẾN HÀNH CÀO THẬT
    print(f"🔗 [{index + 1}/{len(urls)}] Đang xử lý: {url}")
    try:
        # Kiểm tra sống còn của driver
        try:
            _ = driver.current_url
        except:
            init_driver()
            driver.maximize_window()

        data = scrape_place(url)
        data['url_source'] = url
        
        # --- LƯU FILE JSON ---
        file_exists = os.path.exists(OUTPUT_FILE)
        with open(OUTPUT_FILE, "a", encoding="utf-8") as f_out:
            if not file_exists:
                f_out.write("[\n")
            else:
                f_out.write(",\n")
            
            f_out.write(json.dumps(data, ensure_ascii=False, indent=4))

        all_results.append(data)
        print(f"   ✔️ Đã lưu xong {len(data.get('reviews', []))} reviews.")
        
    except Exception as e:
        print(f"   ⚠️ Lỗi: {e}")
        # Nếu lỗi liên quan đến cửa sổ, ép reset driver cho link sau
        if "window" in str(e).lower():
            init_driver()
            driver.maximize_window()
    
    current_batch_count += 1
    # Nghỉ giữa các link (nên để lâu một chút để tránh bị block)
    time.sleep(random.uniform(2, 4))

# 3. KẾT THÚC
if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "a", encoding="utf-8") as f_out:
        f_out.write("\n]")

try: driver.quit()
except: pass
print(f"\n✨ XONG! Kiểm tra file: {OUTPUT_FILE}")

✅ Đã tải 100 URLs.
🚀 Đang khởi tạo trình duyệt...
   🌐 Đang khởi tạo cửa sổ Chrome mới...
🔥 Đang chạy Warm-up cho trình duyệt: https://www.google.com/maps/place/HiHi+TomYum+-+B%C3%A0n+C%E1%BB%9D/@10.7737595,106.6862385,17z/data=!3m1!4b1!4m6!3m5!1s0x31752fdc8d435327:0x252f09bf25e839d4!8m2!3d10.7737595!4d106.6862385!16s%2Fg%2F11s1sx90kn?authuser=0&hl=en&entry=ttu&g_ep=EgoyMDI2MDMzMC4wIKXMDSoASAFQAw%3D%3D
   ⏳ Đang đợi Google Maps render đủ thông tin (2s)...
   ✅ Trình duyệt đã 'nóng', bắt đầu cào thật...
🔗 [1/100] Đang xử lý: https://www.google.com/maps/place/Qu%C3%A1n+%C7%8En+H%C3%B9ng+K%C3%BD/data=!4m7!3m6!1s0x31752efa64fcabf7:0x455fbbc7d3651107!8m2!3d10.7524705!4d106.6658192!16s%2Fg%2F11gdgxn5tq!19sChIJ96v8ZPoudTERBxFl08e7X0U?authuser=0&hl=en&rclk=1
   ⏳ Đang đợi Google Maps render đủ thông tin (2s)...
   ✔️ Đã lưu xong 50 reviews.
🔗 [2/100] Đang xử lý: https://www.google.com/maps/place/B%C3%92+N%C6%AF%E1%BB%9ANG+NHI+%7C+TH%E1%BB%8AT+B%C3%92+%C6%AF%E1%BB%9AP+S%E1%BA%B4N+QU%E1%BA%ACN+5